# Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
# Where watsonx.governance is hosted: "cpd" (Cloud Pak for Data) or "aws" (watsonx.governance SaaS on AWS)
DEPLOYMENT_TYPE = "aws"

# if DEPLOYMENT_TYPE == "cpd"
CPD_URL = ""
CPD_USERNAME = ""
CPD_PASSWORD = ""
CPD_API_KEY=""
WOS_SERVICE_INSTANCE_ID = "" # If None, default instance would be used
WML_CREDENTIALS = {
        "url": CPD_URL,
        "username": CPD_USERNAME,
        "password" : CPD_PASSWORD,
        "instance_id": 'openshift',
        "apikey": CPD_API_KEY,
}

# if DEPLOYMENT_TYPE == "aws"
# AWS-hosted watsonx.governance uses MCSP v2 auth (not IBM Cloud IAM).
# AWS_ACCOUNT_ID scopes the token; AWS_WOS_SERVICE_INSTANCE_ID is the watsonx.governance instance - they are different IDs.
# To look up the instance ID: GET {MCSP_URL}/api/2.0/accounts/{AWS_ACCOUNT_ID}/services
AWS_API_KEY = os.getenv("AWS_API_KEY")
AWS_ACCOUNT_ID = os.getenv("AWS_ACCOUNT_ID")
AWS_WOS_SERVICE_INSTANCE_ID = os.getenv("AWS_WOS_SERVICE_INSTANCE_ID")
MCSP_URL = os.getenv("MCSP_URL")
AWS_WOS_SERVICE_URL = os.getenv("AWS_WOS_SERVICE_URL")

#if ollama is used
OLLAMA_HOST=""

# if watsonx.ai model is used
CLOUD_API_KEY=""
IAM_URL=os.getenv("IAM_URL")
watsonxai_project_id=""

# if anthropic model is used
ANTHROPIC_API_KEY=os.getenv("ANTHROPIC_API_KEY")


In [ ]:
# @hidden_cell
# The project token is an authorization token that is used to access project resources like data sources, connections, and used by platform APIs.
# insert project token

# Define supported model provider inference functions
- by default below, sample functions are provided for watsonx.ai, ollama and anthropic. you can define for other providers as well

In [3]:
import time 

def generate_response_ollama(row, model_id):
        complaint_text = row["input"]
        
        # Make the POST request
        st = time.time()
        response = requests.post(
            f'https://{OLLAMA_HOST}/api/chat',
            headers={'Content-Type': 'application/json'},
            json={
                'model': model_id,
                'messages': [
                    {
                        'role': 'user',
                        'content': f"""Please summarize the complaint in 10 words or less
    
    What is the following complaint about?
    
    ---
    {complaint_text}
    ---"""
                    }
                ],
                'stream': False, 
                'think': False
            },
            timeout=300  # 5 minutes timeout for OpenShift
        )
        time_taken = time.time() - st
    
        response = response.json()
        # print(response)
        
        response_reshape = {
            "generated_text": response["message"]["content"], 
            "generated_token_count": response["eval_count"], 
            "input_token_count": response["prompt_eval_count"], 
            "response_time": time_taken*1000, 
            "complaint_text": row["input"], 
            "output": row["output"]
        }

        return response_reshape

import requests

def generate_access_token_cloud():
    headers = {}
    headers["Content-Type"] = "application/x-www-form-urlencoded"
    headers["Accept"] = "application/json"
    data = {
        "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
        "apikey": CLOUD_API_KEY,
        "response_type": "cloud_iam",
    }
    response = requests.post(IAM_URL + "/identity/token", data=data, headers=headers)
    json_data = response.json()
    iam_access_token = json_data["access_token"]
    return iam_access_token


def generate_answer_watsonxai(row, model_id):
    complaint_text = row["input"]
    url = "https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2023-05-29"
    access_token = generate_access_token_cloud()
    body = {
    	"input": f"""Summarize the complaint in 10 words or less

What is the following complaint about?

---
{complaint_text}
---""",
    	"parameters": {
    		"decoding_method": "greedy",
    		"max_new_tokens": 2047,
    		"min_new_tokens": 0,
    		"repetition_penalty": 1
    	},
    	"model_id": model_id,
    	"project_id": watsonxai_project_id
    }
    
    headers = {
    	"Accept": "application/json",
    	"Content-Type": "application/json",
    	"Authorization": f"Bearer {access_token}"
    }

    st = time.time()
    response = requests.post(
    	url,
    	headers=headers,
    	json=body
    )
    time_taken = time.time() - st
    
    if response.status_code != 200:
    	raise Exception("Non-200 response: " + str(response.text))

    data = response.json()
    # print(data)
    response_reshape = {
            "generated_text": data['results'][0]['generated_text'], 
            "generated_token_count": data['results'][0]['generated_token_count'], 
            "input_token_count": data['results'][0]['input_token_count'], 
            "response_time": time_taken*1000, 
            "complaint_text": row["input"], 
            "output": row["output"]
            # "generated_token_count": response["eval_count"], 
            # "input_token_count": response["prompt_eval_count"], 
            # "response_time": ["total_duration"]/1e6
    }
    return response_reshape


# !pip install anthropic
import anthropic

def generate_response_anthropic(row, model_id):
    complaint_text = row["input"]
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    st = time.time()
    message = client.messages.create(
        model=model_id,
        max_tokens=256,
        messages=[
            {
                "role": "user",
                "content": f"""Summarize the complaint in 10 words or less

What is the following complaint about?

---
{complaint_text}
---"""
            }
        ],
    )
    time_taken = time.time() - st

    response_reshape = {
            "generated_text": message.content[0].text, 
            "generated_token_count": message.usage.output_tokens, 
            "input_token_count": message.usage.input_tokens, 
            "response_time": time_taken*1000, 
            "complaint_text": row["input"], 
            "output": row["output"]
    }
    return response_reshape


# Define Inference & Monitoring Class

In [4]:
import requests
import os
import pandas as pd
from tqdm import tqdm
import csv
import json
from ibm_watson_openscale.supporting_classes.payload_record import PayloadRecord

class BatchInference():
    def __init__(
            self,
            provider_id, 
            model_id, 
            dataset, 
            pta_id, 
            stage,
            project_id,
            space_id,
            label_column,
            context_fields,
            question_field,
            problem_type,
            input_data_type,
            monitors,
            deployment_id,
            wos_client, 
):
        self.provider_id = provider_id #tmp
        self.model_id = model_id #tmp
        self.dataset_name = dataset
        self.dataset = pd.read_csv(dataset)#.iloc[:2, :] #tmp
        self.label_column = label_column
        self.problem_type = problem_type
        self.context_fields = context_fields
        self.question_field = question_field
        self.input_data_type = input_data_type
        self.inference_functions = {

            "ollama":    generate_response_ollama, 
            "watsonx.ai": generate_answer_watsonxai,
            "anthropic": generate_response_anthropic
            
        }
        self.monitors = monitors
        self.wos_client = wos_client
        self.pta_id = pta_id
        self.stage = stage
        self.project_id=project_id
        self.space_id=space_id
        self.deployment_id=deployment_id
        self.mrm_monitor_instance_id, self.payload_data_set_id, self.feedback_data_set_id, self.model_health_ds_id = self.configure_monitors()

    def generate_responses(self, evaluate_in_openscale=True):
        responses = []
        for row_idx in tqdm(range(len(self.dataset))):
            response = self.inference_functions[self.provider_id](self.dataset.iloc[row_idx,:], model_id=self.model_id)
            responses.append(response)
            time.sleep(3)

        
        ### add snippet to send payloads & evaluate in watsonx.governance
        if evaluate_in_openscale:
            test_data_set_name = "data"
            content_type = "multipart/form-data"
            body = {
                "generated_text": "generated_text",
                "generated_token_count": "generated_token_count",
                "input_token_count": "input_token_count",
                "response_time": "response_time", 
                "complaint_text": "complaint_text", 
                "output": "output"
            }

            with open("body.json", "w") as f:
                json.dump(body, f)

            responses_df = pd.DataFrame.from_dict(responses)
            test_data_path = f"{self.dataset_name}_with_output.csv"
            responses_df.to_csv(test_data_path, index=False)

            # print(responses_df)

            # import csv

            test_data_content = []

            feedback_fields = ["complaint_text", "output", "generated_text"]
            payload_fields = ["generated_text", "generated_token_count", "input_token_count", "response_time", "complaint_text"]
            
            feedback_data_content = []
            pl_data = []
            
            from ibm_watson_openscale.supporting_classes.payload_record import PayloadRecord
            
            pl_records = []
            
            from ibm_watson_openscale.supporting_classes.payload_record import PayloadRecord
        

            # subscription.payload_logging.store(records=pl_records)
            
            # payload_data = [
            #     {
            #         "fields": payload_fields,
            #         "values": [[row["generated_text"], row["generated_token_count"], row["input_token_count"], row["response_time"], row["complaint_text"]] for row in pl_data]
            #     }
            # ]
            if self.stage != "production":
                if self.space_id is not None:
                    eval_response  = wos_client.monitor_instances.mrm.evaluate_risk(
                        monitor_instance_id=self.mrm_monitor_instance_id,
                        test_data_set_name=test_data_set_name, 
                        test_data_path=test_data_path,
                        content_type=content_type,
                        # body="body.json",
                        space_id=self.space_id,
                        includes_model_output=False,
                        background_mode=False
                    )
                else:
                    eval_response  = wos_client.monitor_instances.mrm.evaluate_risk(
                        monitor_instance_id=self.mrm_monitor_instance_id,
                        test_data_set_name=test_data_set_name, 
                        test_data_path=test_data_path,
                        content_type=content_type,
                        # body="body.json",
                        project_id=self.project_id,
                        includes_model_output=False,
                        background_mode=False
                    )
    
                print(eval_response)

            else:
                pl_records = []
                feedback_data_content = []
                
                with open(test_data_path, 'r') as csv_file:
                    csv_reader = csv.DictReader(csv_file)
                    rows = list(csv_reader)
        
                    # response  = wos_client.monitor_instances.mrm.get_risk_evaluation(self.mrm_monitor_instance_id, space_id=self.space_id).result._to_dict()
                    # records_to_patch = sorted(
                    #     wos_client.data_sets.get_list_of_records(data_set_id=feedback_test[0]['metadata']['id'], start=response["entity"]["parameters"]["evaluation_start_time"], end=response["entity"]["status"]["completed_at"]).result["records"], 
                    #     key = lambda x: x["entity"]["values"]["scoring_timestamp"]) 
                    # record_ids = [i["entity"]["values"]["record_id"] for i in records_to_patch]
                    
                    for row in rows:
                        # wos_client.data_sets.patch_records(
                        #     data_set_id=self.model_health_ds_id,
                        #     patch_document=[
                        #           {
                        #             "op": "replace",
                        #             "path": "/records/{}/values/response_time".format(rid),
                        #             "value": row["response_time"]
                        #           },
                        #     ]
                        # )
                        # Feedback table
                        result_row = [row[key] for key in feedback_fields if key in row]
                        feedback_data_content.append(result_row)
                
                        # Payload record - batch format
                        request = {
                            "fields": ["complaint_text"],
                            "values": [[str(row["complaint_text"])]]
                        }
                        # request = {
                        #     "parameters": {
                        #         "template_variables": {
                        #             "complaint_text": row["complaint_text"]
                        #         }
                        #     }
                        # }
                    
                        response = {
                            "fields": ["generated_text", "generated_token_count", "input_token_count"],
                            "values": [
                                [row["generated_text"], int(row["generated_token_count"]), int(row["input_token_count"])]
                            ]
                        }
                        # response = {
                        #     "results": [
                        #         {"generated_text": row["generated_text"], "generated_token_count": row["generated_token_count"],"input_token_count": row["input_token_count"] }
                        #     ]
                        # }
                    
                        record = PayloadRecord(
                            request=request,
                            response=response,
                            response_time=int(float(row["response_time"]))  # or aggregate as needed
                        )
                        pl_records.append(record)
                
                feedback_data = [
                    {
                        "fields": feedback_fields,
                        "values": feedback_data_content
                    }
                ]

                print("Performing explicit payload logging.")
                wos_client.data_sets.store_records(
                    data_set_id=self.payload_data_set_id, request_body=pl_records, background_mode=False
                )
                time.sleep(5)
                pl_records_count = wos_client.data_sets.get_records_count(self.payload_data_set_id)
                print("Number of records in the payload logging table: {}".format(pl_records_count))
    
                wos_client.data_sets.store_records(data_set_id=self.feedback_data_set_id, request_body=feedback_data,background_mode=False)
                time.sleep(5)
                fb_records_count = wos_client.data_sets.get_records_count(self.feedback_data_set_id)
                # Adding time delay to enable drift
                time.sleep(10)
                print("Number of records in the feedback logging table: {}".format(fb_records_count))
                # # eval_response = wos_client.monitor_instances.mrm.evaluate_risk(monitor_instance_id=self.mrm_monitor_instance_id,
                # #                                         # body = "body.json",
                # #                                         body = {},
                # #                                         # space_id = self.space_id,
                # #                                         # project_id = self.project_id,
                # #                                         content_type="application/json",
                # #                                         background_mode = False)

                response  = wos_client.monitor_instances.mrm.evaluate_risk(monitor_instance_id=self.mrm_monitor_instance_id, 
                                                    body = body,
                                                    space_id = self.space_id,
                                                    evaluation_tests = ["generative_ai_quality", "model_health", "drift_v2"],
                                                    background_mode = False)
                

            # run_response = wos_client.monitor_instances.run( #'drift_v2',
            #     monitor_instance_id=self.mrm_monitor_instance_id,
            #     background_mode=False, 
            #     parameters={
            #         'model_health': [
            #       # # 'generative_ai_quality',
            #       # # 'generative_ai_security',
            #       'average_api_latency'], 
            #        # 'generative_ai_quality': [
            #        #     "rouge_score"
            #        # ]
            #     'evaluation_tests': [
            #                   'generative_ai_quality',
            #                   'generative_ai_security', 
            #                   'model_health']
            #     }
            # )

            # # test_data_path_json = "test_data_json.json"
            # # with open(test_data_path_json, "w") as f:
            # #     json.dump(pl_records, f)


            # response.result.to_dict()
        ### add snippet to send payloads & evaluate in watsonx.governance
            
        return responses

    def configure_monitors(self):
        try:
            if space_id is not None:
                mrm_monitor_instance_id = wos_client.monitor_instances.mrm.get_prompt_setup(prompt_template_asset_id=self.pta_id, space_id=self.space_id, deployment_id=self.deployment_id).result._to_dict()["mrm_monitor_instance_id"]
            else:
                mrm_monitor_instance_id = wos_client.monitor_instances.mrm.get_prompt_setup(prompt_template_asset_id=self.pta_id, project_id=self.project_id).result._to_dict()["mrm_monitor_instance_id"]
        except:
            if self.space_id is not None:
                response = wos_client.monitor_instances.mrm.execute_prompt_setup(
                    prompt_template_asset_id=self.pta_id, 
                    space_id=self.space_id,
                    label_column=self.label_column,
                    context_fields = self.context_fields,     
                    question_field = self.question_field,     
                    operational_space_id=self.stage, 
                    problem_type=self.problem_type,
                    input_data_type=self.input_data_type, 
                    supporting_monitors=self.monitors, 
                    deployment_id=self.deployment_id,
                    background_mode=False
                )
            else: 
                # response = wos_client.monitor_instances.mrm.execute_prompt_setup(
                #     prompt_template_asset_id=self.pta_id, 
                #     project_id=self.project_id,
                #     label_column=self.label_column,
                #     context_fields = self.context_fields,     
                #     question_field = self.question_field,     
                #     operational_space_id=self.stage, 
                #     problem_type=self.problem_type,
                #     input_data_type=self.input_data_type, 
                #     supporting_monitors=self.monitors, 
                #     background_mode=False
                # )
                response = wos_client.monitor_instances.mrm.execute_prompt_setup(
                    prompt_template_asset_id=self.pta_id, 
                    project_id=self.project_id,
                    label_column=self.label_column,
                    context_fields = self.context_fields,     
                    question_field = self.question_field,     
                    operational_space_id=self.stage, 
                    problem_type=self.problem_type,
                    input_data_type=self.input_data_type, 
                    supporting_monitors=self.monitors, 
                    deployment_id=self.deployment_id,
                    background_mode=False
                )
            result = response.result
            result.to_dict()
            
            if space_id is not None:
                mrm_monitor_instance_id = wos_client.monitor_instances.mrm.get_prompt_setup(prompt_template_asset_id=self.pta_id, space_id=self.space_id, deployment_id=self.deployment_id).result._to_dict()["mrm_monitor_instance_id"]
            else:
                mrm_monitor_instance_id = wos_client.monitor_instances.mrm.get_prompt_setup(prompt_template_asset_id=self.pta_id, project_id=self.project_id).result._to_dict()["mrm_monitor_instance_id"]

     
        ds = wos_client.data_sets.list(target_target_id=wos_client.subscriptions.list(asset_asset_id=pta_id).result._to_dict()['subscriptions'][0]['metadata']['id']).result._to_dict()['data_sets']
        pldsid = [i["metadata"]["id"] for i in ds if i["entity"]['type'] == 'payload_logging'][0]
        fbdsid = [i["metadata"]["id"] for i in ds if i["entity"]['type'] == 'feedback'][0]
        mhdsid = [i["metadata"]["id"] for i in ds if i["entity"]['type'] == 'model_health'][0]
        return mrm_monitor_instance_id, pldsid, fbdsid, mhdsid

# Execute inference

In [9]:
label_column = "output"
context_fields = []
question_field = "complaint_text"
operational_space_id = "development"
problem_type= "summarization"
input_data_type= "unstructured_text"


monitors = {
    "generative_ai_quality": {
        "parameters": {
            "min_sample_size": 1,
            "metrics_configuration": {
                "sentence_similarity": {},
                "text_quality": {},
                "rouge_score": {},
                "meteor": {},
                "flesch": {},
                "sari": {},
                "pii": {},
                "pii_input": {}
            }
        }
    },

    "model_health": {
        "parameters": {
            "min_sample_size": 1,
            "metrics_configuration": {
            # }
        }
    },
}}

dataset = "complaints_dataset_3.csv"

In [10]:
# Loads the dataset from the CPD project's data assets. On AWS, place the CSV next to this notebook instead.
if DEPLOYMENT_TYPE == "cpd":
    import itc_utils.flight_service as itcfs

    nb_data_request = {
        'data_name': dataset,
        'interaction_properties': {
            #'row_limit': 500,
            'infer_schema': 'true',
            'infer_as_varchar': 'false'
        }
    }
    flight_descriptor = itcfs.get_flight_descriptor(nb_data_request=nb_data_request)

    flightClient = itcfs.get_flight_client()
    flightInfo = flightClient.get_flight_info(flight_descriptor)

    df_0 = itcfs.read_pandas_and_concat(flightClient, flightInfo, timeout=240)
    df_0.to_csv(dataset)
elif DEPLOYMENT_TYPE == "aws":
    wslib.download_file(dataset)


In [11]:
from ibm_cloud_sdk_core.authenticators import CloudPakForDataAuthenticator, MCSPV2Authenticator

from ibm_watson_openscale import *
from ibm_watson_openscale.supporting_classes.enums import *
from ibm_watson_openscale.supporting_classes import *

if DEPLOYMENT_TYPE == "cpd":
    authenticator = CloudPakForDataAuthenticator(
        url=CPD_URL,
        username=CPD_USERNAME,
        password=CPD_PASSWORD,
        disable_ssl_verification=True
    )
    wos_client = APIClient(
        service_url=CPD_URL,
        authenticator=authenticator,
        service_instance_id=WOS_SERVICE_INSTANCE_ID
    )
elif DEPLOYMENT_TYPE == "aws":
    authenticator = MCSPV2Authenticator(
        apikey=AWS_API_KEY,
        url=MCSP_URL,
        scope_collection_type="accounts",
        scope_id=AWS_ACCOUNT_ID
    )
    wos_client = APIClient(
        service_url=AWS_WOS_SERVICE_URL,
        authenticator=authenticator,
        service_instance_id=AWS_WOS_SERVICE_INSTANCE_ID
    )
else:
    raise ValueError(f"Unknown DEPLOYMENT_TYPE: {DEPLOYMENT_TYPE}")

print(wos_client.version)


3.1.8


# Workspace & asset specific evaluations

## Anthropic (development)

In [12]:
pta_id = "01a0fd2b-d4f4-7125-a568-bb02e50c42d2"
project_id = "40ba057e-5f17-421e-9616-a1f9b437b5ed"
space_id = None
stage = "development"
provider_id="anthropic"
model_id="claude-haiku-4-5"
data_mart_id = wos_client.service_instance_id

# Space Mapping
# wos_client.wos.add_instance_mapping(                
#                     service_instance_id=data_mart_id,
#                     project_id=project_id
#                  )

inference_obj = BatchInference(
    provider_id, 
    model_id, 
    dataset, 
    pta_id, 
    stage,
    project_id,
    space_id,
    label_column,
    context_fields,
    question_field,
    problem_type,
    input_data_type,
    monitors,
    None,
    wos_client)

responses = inference_obj.generate_responses()


This method will be deprecated in the next release and be replaced by wos_client.wos.get_prompt_setup() method


100%|██████████| 91/91 [05:45<00:00,  3.80s/it]





 Waiting for risk evaluation of MRM monitor 01a0fd59-4161-7f59-9aac-a4aa0fb5d4ce 




upload_in_progress
running.....
finished

---------------------------------------
 Successfully finished evaluating risk 
---------------------------------------


{
    "result": "HTTP response",
    "headers": {
        "_store": {
            "date": [
                "Date",
                "Fri, 02 Oct 2026 18:04:16 GMT"
            ],
            "content-type": [
                "Content-Type",
                "application/json"
            ],
            "transfer-encoding": [
                "Transfer-Encoding",
                "chunked"
            ],
            "connection": [
                "Connection",
                "keep-alive"
            ],
            "server": [
                "server",
                "cloudflare"
            ],
            "x-content-type-options": [
                "x-content-type-options",
                "nosniff"
            ],
            "cache-contr

In [ ]:
pta_id = "01a0fd63-f230-73f3-91f5-d4d52554e670"
# project_id = None
project_id = None
# space_id = None
space_id="7472c9f5-eaa3-4e3f-850a-76c05c2f8f97"
stage = "pre_production"
provider_id="anthropic"
model_id="claude-haiku-4-5"
deployment_id=None
data_mart_id=wos_client.service_instance_id

# Space Mapping
# wos_client.wos.add_instance_mapping(                
#                     service_instance_id=data_mart_id,
#                     project_id=project_id
#                  )

inference_obj = BatchInference(
    provider_id, 
    model_id, 
    dataset, 
    pta_id, 
    stage,
    project_id,
    space_id,
    label_column,
    context_fields,
    question_field,
    problem_type,
    input_data_type,
    monitors,
    None,
    wos_client)

responses = inference_obj.generate_responses()


This method will be deprecated in the next release and be replaced by wos_client.wos.get_prompt_setup() method


  1%|          | 1/91 [00:03<05:53,  3.93s/it]

# add more monitors for production

In [ ]:
monitors = {
    "generative_ai_quality": {
        "parameters": {
            "min_sample_size": 1,
            "metrics_configuration": {
                "sentence_similarity": {},
                "text_quality": {},
                "rouge_score": {},
                "meteor": {},
                "flesch": {},
                "sari": {},
                "pii": {},
                "pii_input": {}
            }
        }
    },

    "model_health": {
        "parameters": {
            "min_sample_size": 1,
            "metrics_configuration": {
            # }
        }
    }},
    "drift_v2": {
        "parameters": {
            "min_samples": 10,
            "train_archive": True
        }
    }
}

In [ ]:
# wos_client.monitor_instances.mrm.get_prompt_setup(prompt_template_asset_id=pta_id, space_id=space_id, deployment_id=deployment_id).result._to_dict()#["mrm_monitor_instance_id"]

In [ ]:
pta_id = "01a0fd6b-65b8-736c-82c7-27c15266dfbf"
# project_id = None
project_id = None
# space_id = None
space_id="50de2fbe-4e67-45f4-bcdb-f6bd5703e389"
stage = "production"
provider_id="anthropic"
model_id="claude-haiku-4-5"
deployment_id=None
data_mart_id=wos_client.service_instance_id

# # Space Mapping
# wos_client.wos.add_instance_mapping(                
#                     service_instance_id=data_mart_id,
#                     space_id=space_id
#                  )

inference_obj = BatchInference(
    provider_id, 
    model_id, 
    dataset, 
    pta_id, 
    stage,
    project_id,
    space_id,
    label_column,
    context_fields,
    question_field,
    problem_type,
    input_data_type,
    monitors,
    deployment_id,
    wos_client)

responses = inference_obj.generate_responses()